## <u>Import Libraries</u>

In [1]:
import gymnasium as gym
import numpy as np
import random
import time

---
## <u>Explore the Environment</u>

Taxi state: a single encoded integer standing in for a 4 part combination: (taxi_row, taxi_col, passenger_location, destination). 25 taxi positions * 5 passenger locations (4 pickup points plus "in taxi") * 4 destinations = 500 total states.

Actions: 6 total, not 4 like Cliff Walking. 0 to 3 are movement (down, up, right, left), 4 is pickup, 5 is dropoff.

Rewards: -1 per step by default, +20 for a correct dropoff, -10 for an illegal pickup/dropoff attempt (e.g. calling dropoff when no passenger is in the taxi). Three distinct reward values here, not two like Cliff Walking.

Episode length cap: Gymnasium wraps this environment in a TimeLimit wrapper capping episodes at 200 steps. Hitting that cap sets `truncated = True`, ending the episode whether or not the task was actually solved. This is a built in safety limit, not something in this code, and it explains episodes that end at exactly 200 steps with a deeply negative reward: the agent simply ran out of steps before completing a delivery.

In [2]:
Q_values = np.zeros((500,6))

env = gym.make("Taxi-v4")

print(env)
print(env.observation_space.n)  # total 500 states
print(env.action_space.n)       # total actions possible, 6
starting_state, extra = env.reset()
print(starting_state)  # the starting position, 414

gamma = 0.99
alpha = 0.5
epsilon = 0.1
episodes = 500
terminated = False
truncated = False

def epsilon_greedy(current_state):

    if random.random() < epsilon:
        return env.action_space.sample()          # explore: random action
    else:
        return np.argmax(Q_values[current_state])   # exploit: best known action for this state

<TimeLimit<OrderEnforcing<PassiveEnvChecker<TaxiEnv<Taxi-v4>>>>>
500
6
286


---
## <u>SARSA Training</u>

Same algorithm as Cliff Walking, unchanged: on policy, carries `next_action` forward every iteration, updates using `Q(s',a')` for the action the policy will actually take next. Only the table shape and the environment itself are different here.

In [3]:
start_time = time.perf_counter()

for episode in range(episodes):

    terminated = False
    truncated = False

    total_rewards = 0
    total_steps = 0

    # every 50th episode gets a renderable (visual) environment, every other episode a normal one
    if episode % 50 == 0:
        env = gym.make("Taxi-v4", render_mode = "human")
    else:
        env = gym.make("Taxi-v4")

    curr_state, extra = env.reset()
    curr_action = epsilon_greedy(curr_state)

    while not (truncated or terminated):
        next_state, reward, terminated, truncated, extra = env.step(curr_action)
        next_action = epsilon_greedy(next_state)

        # SARSA update: uses Q(s',a'), the actual next action the policy will take, exploration included
        td_target = reward + gamma * Q_values[next_state, next_action]
        td_error = td_target - Q_values[curr_state, curr_action]
        Q_values[curr_state, curr_action] = Q_values[curr_state, curr_action] + alpha * td_error

        total_rewards = total_rewards + reward
        total_steps = total_steps + 1

        curr_state = next_state
        curr_action = next_action

    if episode % 50 == 0:
        print(f"for episode {episode+1}/{episodes} --------> total rewards = {total_rewards} -------> total steps = {total_steps}")

    env.close()

end_time = time.perf_counter()
print(f"Visual Training finished in {end_time - start_time:.2f} seconds")

for episode 1/500 --------> total rewards = -650 -------> total steps = 200
for episode 51/500 --------> total rewards = -362 -------> total steps = 200
for episode 101/500 --------> total rewards = -118 -------> total steps = 85
for episode 151/500 --------> total rewards = -34 -------> total steps = 37
for episode 201/500 --------> total rewards = -2 -------> total steps = 23
for episode 251/500 --------> total rewards = 7 -------> total steps = 14
for episode 301/500 --------> total rewards = 10 -------> total steps = 11
for episode 351/500 --------> total rewards = -8 -------> total steps = 29
for episode 401/500 --------> total rewards = -2 -------> total steps = 14
for episode 451/500 --------> total rewards = 7 -------> total steps = 14
Visual Training finished in 168.04 seconds


---
## <u>Test the trained policy</u>

Pure exploitation, no epsilon-greedy: `np.argmax(Q_values[curr_state])` always takes the best known action for whichever state the agent is actually in, since this is evaluating the learned policy, not still training it.

In [4]:
env = gym.make("Taxi-v4", render_mode = "human")

total_rewards = 0
total_steps = 0
terminated = False
truncated = False

curr_state, extra = env.reset()

while not (terminated or truncated):

    curr_action = np.argmax(Q_values[curr_state])  # always exploit, no exploration during testing
    next_state, reward, terminated, truncated, extra = env.step(curr_action)

    total_rewards = total_rewards + reward
    total_steps = total_steps + 1

    curr_state = next_state

env.close()
print(f"total reward = {total_rewards} & episode len = {total_steps}")

total reward = 3 & episode len = 18


---
## <u>Findings</u>

Training progression across 500 episodes: episodes 1 and 51 both hit the 200 step cap with deeply negative rewards (-650 and -362), the agent still mostly flailing between illegal pickup/dropoff attempts (-10 each) and wasted movement. Step counts then drop quickly, 85 at episode 101 and 37 at episode 151, and from episode 201 onward almost every printed episode is a short, successful delivery of 11 to 29 steps with rewards around zero or above (episode 301: reward 10 in only 11 steps), a genuine sign of the policy converging toward correct pickup and dropoff behaviour rather than wandering.

The final tested policy (pure exploitation, no exploration) achieved a total reward of **3** in **18 steps**, a successful delivery: the final dropoff step pays +20 instead of -1, so 17 steps at -1 each plus 20 gives exactly 3. This confirms the learned Q-table works outside the training loop's exploration noise, not just during training itself.

Training took 168 seconds, almost all of it from rendering every 50th episode in a window.